In [ ]:
import json
import pathlib

tro_positions_by_task = json.loads(
    pathlib.Path("../metadata/task_object_infos.json").read_text()
)
print(tro_positions_by_task)

In [ ]:
all_tro_z_positions, all_tro_names = zip(
    *sorted(
        [
            (info["pos"][2], name)
            for task_positions in tro_positions_by_task.values()
            for name, info in task_positions.items()
        ]
    )
)
print(all_tro_z_positions)

# Plot the TRO Z positions
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import random

sns.displot(y=all_tro_z_positions, kde=True)
plt.ylim(0, 3)
plt.show()

In [ ]:
bb_infos = [
    ((info["bbmin"][2], info["bbmax"][2]), name)
    for task_ranges in tro_positions_by_task.values()
    for name, info in task_ranges.items()
]
sorted_bb_infos = sorted(bb_infos, key=lambda x: (x[0][0] + x[0][1]) / 2.0)
all_tro_z_ranges, all_tro_names = zip(*sorted_bb_infos)
print(all_tro_z_ranges)
print(all_tro_names)

# Plot the TRO Z ranges
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import random

plt.figure(figsize=(3, 6), dpi=300)

# Use seaborn style for aesthetics
sns.set_style("whitegrid")

# Plot each range as a vertical line
for i, (min_val, max_val) in enumerate(all_tro_z_ranges):
    # Choose x position based on index
    x_pos = i + 1

    # Plot vertical line for the range
    plt.plot(
        [x_pos, x_pos],
        [min_val, max_val],
        # marker='_', markersize=10,
        linewidth=0.1,
        # label=f'Range {i+1}',
        # c="b"
    )

# Customize the plot
plt.title("1D Ranges Visualization", fontsize=15)
plt.xlabel("Range Index", fontsize=12)
plt.ylabel("Range Values", fontsize=12)

# Adjust x-axis to show some padding
plt.xlim(0, len(all_tro_z_ranges) + 1)
plt.ylim(0, 3)

# Show the plot
plt.tight_layout()
plt.show()

In [ ]:
# 2d plot of positions vs distances from occupancy
all_tro_z_positions, all_tro_dists_from_trav, all_tro_names = zip(
    *sorted(
        [
            (info["pos"][2], np.linalg.norm(info["dist_from_trav"]), name)
            for task_positions in tro_positions_by_task.values()
            for name, info in task_positions.items()
        ]
    )
)
print(all_tro_z_positions)
print(all_tro_dists_from_trav)

# Plot the TRO Z positions
from matplotlib.colors import LogNorm
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import random

plt.figure(figsize=(3, 6), dpi=300)
sns.displot(
    x=all_tro_dists_from_trav, y=all_tro_z_positions, cbar=True, binwidth=(0.1, 0.1)
)
# sns.scatterplot(x=all_tro_dists_from_trav, y=all_tro_z_positions, s=10, marker="x")
plt.ylim(0, 3)
plt.ylabel("Height from Floor")
plt.xlim(0, 3)
plt.xlabel("Distance from Nearest\nAccessible Horizontal Position")
plt.show()

In [ ]:
def normalize_to_nearest_axis(angles):
    """
    Normalize angles to their closest axis (0°, 90°, 180°, 270°) by converting them
    to the smallest angle difference.

    For example:
    - 170° becomes -10° (distance to 180°)
    - 80° becomes -10° (distance to 90°)
    - 280° becomes 10° (distance to 270°)

    Parameters:
    -----------
    angles : numpy.ndarray
        Array of angles in degrees

    Returns:
    --------
    numpy.ndarray
        Array of normalized angles in degrees, always between -45° and 45°
    """
    # Ensure angles are between 0 and 360
    angles = np.rad2deg(angles) % 360

    # Calculate distance to each axis
    dist_0_360 = np.minimum(angles, np.abs(angles - 360))
    dist_90 = np.abs(angles - 90)
    dist_180 = np.abs(angles - 180)
    dist_270 = np.abs(angles - 270)

    # Stack all distances
    all_dists = np.vstack([dist_0_360, dist_90, dist_180, dist_270])

    # Find which axis each angle is closest to
    closest_axis = np.argmin(all_dists, axis=0) * 90

    # Calculate the signed difference to the closest axis
    diff = angles - closest_axis

    # Normalize differences to be between -45 and 45
    diff = (diff + 180) % 360 - 180
    diff = np.where(diff > 45, diff - 180, diff)
    diff = np.where(diff < -45, diff + 180, diff)

    return np.deg2rad(diff)


# 2d plot of positions vs distances from occupancy
all_tro_x_positions, all_tro_y_positions = zip(
    *sorted(
        [
            info["dist_from_trav"]
            for task_positions in tro_positions_by_task.values()
            for name, info in task_positions.items()
        ]
    )
)
print(all_tro_x_positions)
print(all_tro_y_positions)

# Plot the TRO Z positions
from matplotlib.colors import LogNorm
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import random

# Convert the X and Y positions to polar coordinates
all_tro_angles = np.arctan2(all_tro_y_positions, all_tro_x_positions)
all_tro_radii = np.linalg.norm([all_tro_x_positions, all_tro_y_positions], axis=0)

# Now change the angles to instead be the angles to the nearest axis
# e.g. 170 degrees is actually -10, 80 degrees is -10, 280 is 10, etc.
all_tro_angles = normalize_to_nearest_axis(all_tro_angles)

# Now convert back to cartesian. We want x+ to be up on this image so we add 90 deg to the angles
all_tro_x_positions = all_tro_radii * np.cos(all_tro_angles + np.pi / 2)
all_tro_y_positions = all_tro_radii * np.sin(all_tro_angles + np.pi / 2)

plt.figure(figsize=(3, 3), dpi=300)
# sns.scatterplot(x=all_tro_x_positions, y=all_tro_y_positions, s=10, marker="x")
sns.displot(
    x=all_tro_x_positions, y=all_tro_y_positions, cbar=True, binwidth=(0.05, 0.05)
)
plt.ylim(-1, 1)
plt.ylabel("Y Position")
plt.xlim(-1, 1)
plt.xlabel("X Position")
plt.show()

In [ ]:
# Lets try to render a robot, too
import omnigibson.utils.urdfpy_utils
import trimesh
import pyrender

urdf_contents = pathlib.Path(
    "/scr/og-docker-data/datasets/assets/models/r1/urdf/r1.urdf"
).read_text()
urdf_contents = urdf_contents.replace(
    "/home/jdw/projects/repo/OmniGibson/omnigibson/data/external_dataset/objects/robot",
    "/scr/og-docker-data/datasets/assets/models",
)
temp_path = pathlib.Path(
    "/scr/og-docker-data/datasets/assets/models/r1/urdf/r1_temp.urdf"
)
temp_path.write_text(urdf_contents)
robot = omnigibson.utils.urdfpy_utils.URDF.load(str(temp_path))
tensor = np.array
knee_bend = np.deg2rad(0)
torso_bend = np.deg2rad(0)
joint_cfg = {
    "torso_joint1": tensor(np.deg2rad(0)),
    "torso_joint2": tensor(np.deg2rad(0)),
    "torso_joint3": tensor(np.deg2rad(0)),
    "torso_joint4": tensor(0.0),
    "left_arm_joint1": tensor(0.0),
    "right_arm_joint1": tensor(0.0),
    "left_arm_joint2": tensor(1.9060),
    "right_arm_joint2": tensor(1.9060),
    "left_arm_joint3": tensor(-0.9910),
    "right_arm_joint3": tensor(-0.9910),
    "left_arm_joint4": tensor(1.5710),
    "right_arm_joint4": tensor(1.5710),
    "left_arm_joint5": tensor(0.9150),
    "right_arm_joint5": tensor(0.9150),
    "left_arm_joint6": tensor(-1.5710),
    "right_arm_joint6": tensor(-1.5710),
    "left_gripper_axis1": tensor(0.0300),
    "left_gripper_axis2": tensor(0.0300),
    "right_gripper_axis1": tensor(0.0300),
    "right_gripper_axis2": tensor(0.0300),
}
vfk = robot.visual_trimesh_fk(cfg=joint_cfg)

scene = pyrender.Scene()
for mesh, transform in vfk.items():
    scene.add(pyrender.Mesh.from_trimesh(mesh), pose=transform)

import os

os.environ["DISPLAY"] = ":1"
camera = pyrender.OrthographicCamera(xmag=1.0, ymag=1.0)

TOP_CAMERA = True

if not TOP_CAMERA:
    camera_position = np.array([0, -5, 0.8])
    camera_z = np.array([0, -1, 0])
    camera_y = np.array([0, 0, 1])
    camera_x = np.cross(camera_y, camera_z)
    camera_pose = np.eye(4)
    camera_pose[:3, 0] = camera_x
    camera_pose[:3, 1] = camera_y
    camera_pose[:3, 2] = camera_z
    camera_pose[:3, 3] = camera_position
else:
    camera_position = np.array([0, 0, 5])
    camera_z = np.array([0, 0, 1])
    camera_y = np.array([1, 0, 0])
    camera_x = np.cross(camera_y, camera_z)
    camera_pose = np.eye(4)
    camera_pose[:3, 0] = camera_x
    camera_pose[:3, 1] = camera_y
    camera_pose[:3, 2] = camera_z
    camera_pose[:3, 3] = camera_position
scene.add(camera, pose=camera_pose)
light = pyrender.DirectionalLight(color=[1.0, 1.0, 1.0], intensity=2.0)
scene.add(light, pose=camera_pose)
r = pyrender.OffscreenRenderer(400, 400)
color, depth = r.render(scene)

# Convert the color into a PIL image where all the white pixels are transparent
import PIL.Image

img = PIL.Image.fromarray(color)
img = img.convert("RGBA")
data = img.getdata()
newData = []
for item in data:
    # Check if the pixel is white (allowing for slight variations)
    if item[0] > 250 and item[1] > 250 and item[2] > 250:
        newData.append((255, 255, 255, 0))  # Full transparency
    else:
        newData.append(item)
img.putdata(newData)
img